### Load Skills — Storyboard Storytelling Pipeline

本 notebook 演示如何加载 Agnes 平台上的 **Skills**，
具体以 `Skills/storyboard-storytelling-pipeline` 为例：

1. 读取 SKILL.md 及关键引用文件，作为 System Prompt 注入
2. 通过 Agnes API (OpenAI 兼容) 与加载了 Skill 的模型对话
3. 让模型按 Skill 定义的管线输出剧本分析、序列地图、分镜策略等

## 0. 加载环境变量

In [1]:
import os
import json

# 从环境变量读取 Agnes API Key
api_key = os.getenv("AGNES_API_KEY")
if not api_key:
    raise ValueError("请先设置环境变量 AGNES_API_KEY")
print("✓ API Key 已加载")

✓ API Key 已加载


## 1. 读取 Skill 内容

Skill 的核心定义位于 `SKILL.md`，同时引用 `references/` 下的详细参考文件。
我们将这些内容读取后合并成 System Prompt 注入模型。

> ⚠ **路径说明**：由于 notebook 在 `test/` 目录下运行，
> 代码会自动计算项目根目录路径，确保能找到 `Skills/` 下的文件。

In [2]:
import os

# ---- 路径修正 ----
# 从当前目录向上查找项目根目录（特征：Skills/ 目录存在）
def _find_project_root():
    _current = os.path.abspath(".")
    while True:
        if os.path.isdir(os.path.join(_current, "Skills")):
            return _current
        _parent = os.path.dirname(_current)
        if _parent == _current:
            raise FileNotFoundError("无法找到项目根目录（特征：Skills/ 目录），请确认 notebook 运行目录正确")
        _current = _parent

PROJECT_ROOT = _find_project_root()

SKILL_BASE = os.path.join(PROJECT_ROOT, "Skills", "storyboard-storytelling-pipeline")
print(f"  SKILL 路径: {SKILL_BASE}")
print(f"  SKILL.md 存在: {os.path.exists(os.path.join(SKILL_BASE, 'SKILL.md'))}")

def read_file_content(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# 1. 读取核心 SKILL.md
skill_md = read_file_content(os.path.join(SKILL_BASE, "SKILL.md"))
print(f"  SKILL.md 成功读取 ({len(skill_md)} 字符)")

# 2. 读取关键引用文件
references = {}
ref_files = [
    "references/floobynooby-core.md",
    "references/workflow.md",
    "references/templates.md",
    "references/night-nuo-case-study.md",
]

for ref in ref_files:
    try:
        ref_path = os.path.join(SKILL_BASE, ref)
        references[ref] = read_file_content(ref_path)
        print(f"  已加载: {ref}")
    except FileNotFoundError:
        print(f"  未找到: {ref} (跳过，不影响主流程)")

print(f"\n引用文件数: {len(references)}")

  SKILL 路径: e:\gitCode\AI应用学习\learn-journal-\AI大模型应用开发工程师\Skills\storyboard-storytelling-pipeline
  SKILL.md 存在: True
  SKILL.md 成功读取 (8599 字符)
  已加载: references/floobynooby-core.md
  已加载: references/workflow.md
  已加载: references/templates.md
  已加载: references/night-nuo-case-study.md

引用文件数: 4


In [3]:
# 构建 System Prompt：将 SKILL.md + 所有引用文件合并

system_prompt = f"""你是 Storyboard Storytelling Pipeline 专家。
请严格按照以下 Skill 定义来指导分镜工作。

====================
## SKILL 核心定义
====================
{skill_md}
"""

# 追加引用文件（作为知识补充）
for ref_name, ref_content in references.items():
    system_prompt += f"\n\n====================\n## 引用: {ref_name}\n====================\n{ref_content}"

print(f"System Prompt 总长度: {len(system_prompt)} 字符")
print(f"Token 估算 (~4字符/token): {len(system_prompt) // 4}")

System Prompt 总长度: 29040 字符
Token 估算 (~4字符/token): 7260


## 2. 通过 Agnes AI 调用 Skill

使用 Requests 库直接调用 Agnes Chat Completions API，
将上一步构建的 System Prompt 注入，让模型以 Storyboard Pipeline 专家身份工作。

In [5]:
import requests

url = "https://apihub.agnes-ai.com/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# 用户输入：一个需要分镜的剧本或场景描述
user_input = """
我有一个短剧剧本：

【场景】废弃工厂，黄昏
【角色】阿杰（男主，25岁，前特种兵），神秘女子（戴面具，身份不明）
【剧情】阿杰追踪线索来到废弃工厂。他发现地上有血迹，顺着血迹走到一台旧机器前。
机器突然启动，神秘女子从阴影中走出。她说："你终于来了。" 阿杰举起枪："面具摘掉。"
女子缓缓摘下面具，露出一张阿杰无比熟悉的脸——是他以为已经死去的姐姐。
阿杰的枪口开始颤抖。
"这不可能……"

请按 Storyboard Pipeline 对这段剧本进行：
1. 剧本解读（dramatic structure）
2. 序列地图（sequence map）
3. 镜头策略（camera strategy）
4. 每场的 shot flow
5. rough thumbnails 任务表
"""

payload = {
    "model": "agnes-2.0-flash",
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ],
    "temperature": 0.7,
    "max_tokens": 16384
}

print("⏳ 正在请求 Agnes API 加载 Skill 并推理...")
response = requests.post(url, headers=headers, data=json.dumps(payload))
result = response.json()

if "choices" in result:
    print("\n" + "="*60)
    print(" AGNES (with Storyboard Pipeline Skill) 输出:")
    print("="*60)
    print(result["choices"][0]["message"]["content"])
else:
    print(" 请求失败:", json.dumps(result, ensure_ascii=False, indent=2))

⏳ 正在请求 Agnes API 加载 Skill 并推理...

 AGNES (with Storyboard Pipeline Skill) 输出:


以下按 Pipeline 前五个阶段输出。内容严格遵循 `storyboard-storytelling-pipeline` 的工作流与模板规范，聚焦故事推进、镜头语言逻辑与可执行性。

---

## Step 1: 剧本解读（Full Script Read）

**戏剧结构核心：**
- **主角与欲望**：阿杰（前特种兵），被“寻找失踪姐姐”的执念驱动，处于高度警觉的狩猎状态。
- **对抗力量**：废弃工厂的环境压迫 + 未知面具女子的心理博弈 + 机器启动带来的物理威胁。
- **信息释放顺序**：环境/线索（血迹）→ 危机触发（机器启动）→ 对峙（台词交锋）→ 情感爆点（面具揭示）→ 状态倒转（枪口颤抖）。
- **高潮定位**：非动作高潮，而是认知与情感高潮。姐姐现身是真正的戏剧重心。
- **钩子与悬念**：姐姐为何活着？面具下是谁？机器与血迹的关联是什么？为后续剧情留出接口。

**不做什么：**
- 不在此阶段设计详细机位或景别切换。
- 不提前解释女子身份或工厂背景，保留视觉悬念。

---

## Step 2: 序列地图（Sequence Drama Map）

```md
## Sequence 1: 废弃工厂·追踪与揭露

戏核：
- 猎杀者被反杀，物理追踪瞬间转为情感崩盘。

这段结束后改变了什么：
- 阿杰从“主动追击”变为“被动承受”，身份从猎人跌落为失控者。
- 观众对“姐姐已死”的既定认知被推翻，悬念性质从“找线索”升级为“认身份”。

观众站位：
- 始终贴合阿杰的动线与视线逻辑，最后半秒抽离为客观三人关系视角。

主要压力来源：
- 前半段：空间幽闭+血迹引导的不安感。
- 后半段：面具摘下的不可逆信息冲击。

镜头总策略：
- 由外向内收。开场保持中全景建立环境呼吸感，中段用机器启动做硬切点，后段压缩景别但保留静止重量，不靠剪辑速度堆情绪。

不要怎么拍：
- 不要用快切掩盖铺垫；不要让女子过早流露情绪；不要在中段插入无意义的空镜或装备特写。
```

---

## Step 3: 每序列镜头策略（Per-Sequence Camera Strategy）



## 3. 多轮对话：继续管线工作

Skill 的管线不是一次对话就能完成的。我们可以继续追问后续环节。

In [ ]:
# 继续对话：针对上一次输出提出更深的问题
# 注意：这里使用独立的 API 调用（无上下文记忆），
# 实际生产环境中应维护 messages 列表来保留对话历史

follow_up = (
    "基于你刚才的分析，请重点输出第4场（面具摘掉）的：\n"
    "1. key panel 设计（关键帧构图描述）\n"
    "2. clean-pass 执行规范\n"
    "3. 最终交付说明"
)

payload = {
    "model": "agnes-2.0-flash",
    "messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input + "\n\n---" + follow_up}
    ],
    "temperature": 0.7,
    "max_tokens": 4096
}

print("⏳ 继续请求...")
response = requests.post(url, headers=headers, data=json.dumps(payload))
result = response.json()

if "choices" in result:
    print("\n" + "="*60)
    print(" 继续输出:")
    print("="*60)
    print(result["choices"][0]["message"]["content"])
else:
    print(" 请求失败:", json.dumps(result, ensure_ascii=False, indent=2))

## 4. 使用 OpenAI SDK（可选替代方案）

如果安装了 `openai` 库，也可以使用 OpenAI SDK 调用，代码更简洁。

In [ ]:
from openai import OpenAI

# 创建 Agnes 客户端
agnes_client = OpenAI(
    api_key=api_key,
    base_url="https://apihub.agnes-ai.com/v1"
)

print("⏳ 使用 OpenAI SDK 调用...")

response = agnes_client.chat.completions.create(
    model="agnes-2.0-flash",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": (
            "请用一句话总结 Storyboard Storytelling Pipeline 的核心工作流程，"
            "以及它和普通 shot list 做分镜最大的区别是什么。"
        )}
    ],
    temperature=0.7,
    max_tokens=1024
)

print("\n" + "="*60)
print(response.choices[0].message.content)
print("="*60)